In [1]:
import os
import json
import urllib.request
import importlib, pathlib, sys
from urllib.parse import urlparse
from datetime import datetime, timezone

import teehr
import pandas as pd
from teehr.evaluation.spark_session_utils import create_spark_session

from teehr import DeterministicMetrics as dm
from teehr import Signatures as s
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf
from teehr import Bootstrappers as bs

from teehr.models.filters import TableFilter

from pyspark.sql import functions as F

from pyspark.sql import DataFrame

import copy
import time

teehr.__version__

'0.7.0'

In [2]:
# spark = create_spark_session(
#     start_spark_cluster=True,
#     executor_instances=64,
#     executor_memory="16g",
#     executor_cores=2,
#     aws_profile="default",
#     pod_template_path=pod_template_path,
#     update_configs={
#         "spark.sql.shuffle.partitions": 1024,
#         "spark.sql.adaptive.coalescePartitions.enabled": "false",
#         "spark.kubernetes.executor.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict": "false",
#         "spark.executorEnv.TEEHR_BOOTSTRAP_ENGINE": "vectorized",
#         "spark.executor.memoryOverhead": "4g",
#     }
# )

spark = create_spark_session()

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS session token from boto3
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


In [3]:
spark.sql("USE iceberg.teehr")

DataFrame[]

In [4]:
spark.sql("""
SELECT *
FROM nwmd_metrics_by_location_v2 
LIMIT 10
""").show()

{"ts": "2026-09-09 10:55:05.323", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `nwmd_metrics_by_location_v2` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o129.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `nwmd_metrics_by_location_v2` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `nwmd_metrics_by_location_v2` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 3 pos 5;
'GlobalLimit 10
+- 'LocalLimit 10
   +- 'Project [*]
      +- 'UnresolvedRelation [nwmd_metrics_by_location_v2], [], false


In [8]:
# List tables
spark.sql("""
SELECT * FROM nwmd_flow_thresholds
WHERE location_id = 'usgs-14171000'
""").show()

+-------------+--------------------+---------+--------+-------------------+-------------------+--------+-----------------+--------------------+
|  location_id|       variable_name|unit_name|n_values|          por_start|            por_end|quantile|  threshold_value|         computed_at|
+-------------+--------------------+---------+--------+-------------------+-------------------+--------+-----------------+--------------------+
|usgs-14171000|streamflow_none_inst|    m^3/s|  221429|2000-10-01 07:00:00|2026-09-08 07:00:00|    0.85|21.54911994934082|2026-09-08 14:07:...|
|usgs-14171000|streamflow_none_inst|    m^3/s|  221429|2000-10-01 07:00:00|2026-09-08 07:00:00|    0.95|44.45744705200195|2026-09-08 14:07:...|
|usgs-14171000|streamflow_none_inst|    m^3/s|  221429|2000-10-01 07:00:00|2026-09-08 07:00:00|    0.99|  89.764404296875|2026-09-08 14:07:...|
+-------------+--------------------+---------+--------+-------------------+-------------------+--------+-----------------+--------------

In [9]:
spark.sql("""
SELECT * FROM fcst_joined_timeseries
LIMIT 10
""").show()


+-------------------+-------------------+-------------------+---------------------+-------------+---------------+--------------------+---------+--------------------+------+--------------------+--------------------+
|     reference_time|         value_time|primary_location_id|secondary_location_id|primary_value|secondary_value|  configuration_name|unit_name|       variable_name|member|          created_at|          updated_at|
+-------------------+-------------------+-------------------+---------------------+-------------+---------------+--------------------+---------+--------------------+------+--------------------+--------------------+
|2026-08-30 00:00:00|2026-09-01 00:00:00|      usgs-16010000|      nwm30-800007054|    1.8292683|           0.37|nwm30_short_range...|    m^3/s|streamflow_15min_...|  NULL|2026-09-08 09:23:...|2026-09-08 09:23:...|
|2026-08-30 12:00:00|2026-09-01 00:00:00|      usgs-16010000|      nwm30-800007054|    1.8292683|           0.22|nwm30_short_range...|    m^

In [11]:
spark.sql("""
SELECT 'thresholds' AS src, collect_set(variable_name) AS vars, collect_set(unit_name) AS units
FROM (SELECT DISTINCT variable_name, unit_name FROM nwmd_flow_thresholds)
""").show()

spark.sql("""
SELECT 'fcst_joined', collect_set(variable_name), collect_set(unit_name)
FROM (SELECT DISTINCT variable_name, unit_name FROM fcst_joined_timeseries)
""").show()

spark.sql("""
SELECT 'primary_ts', collect_set(variable_name), collect_set(unit_name)
FROM (SELECT DISTINCT variable_name, unit_name FROM primary_timeseries)
""").show()

+----------+--------------------+-------+
|       src|                vars|  units|
+----------+--------------------+-------+
|thresholds|[streamflow_none_...|[m^3/s]|
+----------+--------------------+-------+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/srv/conda/envs/notebook/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
INFO:py4j.java_gateway:Search for sockets that match local addr ('127.0.0.1', 58694) and remote addr ('127.0.0.1', 44269)
INFO:py4j.java_gateway:Shutting down matched socket <socket.socket fd=55, family=2, type=1, proto=0, laddr=('127.0.0.1', 46936), raddr=('127.0.0.1', 44269)>
INFO:py4j.clientserver:Close connection s

KeyboardInterrupt: 

In [5]:
spark.sql("""
SELECT 
    count(*)
FROM fcst_joined_timeseries
WHERE configuration_name in ('nwm30_medium_range', 'nwm30_short_range')
""").show(truncate=False)


+----------+
|count(1)  |
+----------+
|9764115437|
+----------+



In [6]:
spark.sql("""
SELECT 
    count(*)
FROM nwmd_metrics_by_location_v2
GROUP BY configuration_name
""").show(truncate=False)

+--------+
|count(1)|
+--------+
|7378365 |
|2686017 |
+--------+



In [7]:
spark.sql("""
SELECT 
    configuration_name,
    collect_set(forecast_lead_time_bin) as forecast_lead_time_bins,
    collect_set(water_year) as water_years,
    collect_set(threshold) as thresholds,
    collect_set(quarter) as quarters,
    collect_set(window_agg) as window_aggs
FROM nwmd_metrics_by_location_v2 
GROUP BY configuration_name
""").show(truncate=False)

+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------------------------+------------------------------------------------------------------------+----------------+
|configuration_name|forecast_lead_time_bins                                                                                                                                               |water_years |thresholds                       |quarters                                                                |window_aggs     |
+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------------------------+------------------------------------------------------------------------+----------------+
|nwm30_medium_range|[P1DT

In [8]:
spark.sql("""
SELECT 
    configuration_name, water_year, threshold, quarter, window_agg, count(*)
FROM nwmd_metrics_by_location_v2
GROUP BY configuration_name, water_year, threshold, quarter, window_agg
ORDER BY configuration_name, water_year, threshold, quarter, window_agg
""").show(truncate=False)

+------------------+----------+---------+-------+----------+--------+
|configuration_name|water_year|threshold|quarter|window_agg|count(1)|
+------------------+----------+---------+-------+----------+--------+
|nwm30_medium_range|2025      |NULL     |NULL   |max       |81654   |
|nwm30_medium_range|2025      |NULL     |NULL   |mean      |81654   |
|nwm30_medium_range|2025      |NULL     |NULL   |min       |81654   |
|nwm30_medium_range|2025      |NULL     |2024-Q4|max       |80587   |
|nwm30_medium_range|2025      |NULL     |2024-Q4|mean      |80587   |
|nwm30_medium_range|2025      |NULL     |2024-Q4|min       |80587   |
|nwm30_medium_range|2025      |NULL     |2025-Q1|max       |80458   |
|nwm30_medium_range|2025      |NULL     |2025-Q1|mean      |80458   |
|nwm30_medium_range|2025      |NULL     |2025-Q1|min       |80458   |
|nwm30_medium_range|2025      |NULL     |2025-Q2|max       |81300   |
|nwm30_medium_range|2025      |NULL     |2025-Q2|mean      |81300   |
|nwm30_medium_range|

In [9]:
spark.stop()